# 第14章 时间序列、采样与周期信号

这个 notebook 用一条极小教学光变曲线，演示不均匀采样、候选周期扫描、最小正弦拟合和相位折叠。

## 学习目标

- 读入并检查时间序列的时间覆盖和误差
- 在有限周期网格上做最小周期搜索
- 理解最佳周期与次优候选的差别
- 计算相位并完成相位折叠
- 建立时间序列分析的最小算法骨架

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from pathlib import Path

DATA_PATH = Path('../../data/small/lightcurve_period_demo.csv')

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = [
        {
            'sample_id': row['sample_id'],
            'time_day': float(row['time_day']),
            'flux_norm': float(row['flux_norm']),
            'flux_error': float(row['flux_error']),
            'split': row['split'],
            'object_class': row['object_class'],
        }
        for row in csv.DictReader(handle)
    ]

times = [row['time_day'] for row in rows]
fluxes = [row['flux_norm'] for row in rows]
errors = [row['flux_error'] for row in rows]

print(f'Loaded {len(rows)} time-series rows from {DATA_PATH.name}')
print('Time span [day]:', (min(times), max(times)))
print('Flux range:', (round(min(fluxes), 3), round(max(fluxes), 3)))
print('Typical error:', round(sum(errors) / len(errors), 3))
print('Python version:', platform.python_version())


In [ ]:
def solve_linear_system(matrix, vector):
    augmented = [row[:] + [value] for row, value in zip(matrix, vector)]
    n = len(augmented)
    for col in range(n):
        pivot = max(range(col, n), key=lambda idx: abs(augmented[idx][col]))
        augmented[col], augmented[pivot] = augmented[pivot], augmented[col]
        pivot_value = augmented[col][col]
        if abs(pivot_value) < 1e-12:
            raise ValueError('Singular matrix in period fit')
        for j in range(col, n + 1):
            augmented[col][j] /= pivot_value
        for row_idx in range(n):
            if row_idx == col:
                continue
            factor = augmented[row_idx][col]
            for j in range(col, n + 1):
                augmented[row_idx][j] -= factor * augmented[col][j]
    return [augmented[i][-1] for i in range(n)]

def fit_period(period, samples):
    omega = 2.0 * math.pi / period
    normal = [[0.0] * 3 for _ in range(3)]
    rhs = [0.0, 0.0, 0.0]
    for row in samples:
        s = math.sin(omega * row['time_day'])
        c = math.cos(omega * row['time_day'])
        design = [1.0, s, c]
        weight = 1.0 / (row['flux_error'] ** 2)
        for i in range(3):
            rhs[i] += weight * design[i] * row['flux_norm']
            for j in range(3):
                normal[i][j] += weight * design[i] * design[j]
    c0, a_sin, b_cos = solve_linear_system(normal, rhs)
    chi2 = 0.0
    fitted = []
    for row in samples:
        model = c0 + a_sin * math.sin(omega * row['time_day']) + b_cos * math.cos(omega * row['time_day'])
        residual = row['flux_norm'] - model
        chi2 += (residual / row['flux_error']) ** 2
        fitted.append(model)
    amplitude = math.sqrt(a_sin ** 2 + b_cos ** 2)
    phase_angle = math.atan2(b_cos, a_sin)
    return {
        'period': period,
        'chi2': chi2,
        'offset': c0,
        'a_sin': a_sin,
        'b_cos': b_cos,
        'amplitude': amplitude,
        'phase_angle': phase_angle,
        'fitted': fitted,
    }


In [ ]:
candidate_periods = [1.6 + 0.05 * i for i in range(29)]
solutions = [fit_period(period, rows) for period in candidate_periods]
solutions.sort(key=lambda item: item['chi2'])

best = solutions[0]
print('Top candidate periods by chi-square:')
for candidate in solutions[:5]:
    print(
        '  P={:.2f} day chi2={:.3f} amplitude={:.3f}'.format(
            candidate['period'], candidate['chi2'], candidate['amplitude']
        )
    )


In [ ]:
def phase_fold(samples, period):
    folded = []
    for row in samples:
        phase = (row['time_day'] / period) % 1.0
        folded.append((phase, row['flux_norm']))
    folded.sort(key=lambda item: item[0])
    return folded

folded = phase_fold(rows, best['period'])
print('Best period [day]:', round(best['period'], 3))
print('Best-fit offset:', round(best['offset'], 4))
print('Best-fit amplitude:', round(best['amplitude'], 4))
print('First five phase-folded points:')
for phase, flux in folded[:5]:
    print('  phase={:.3f}, flux={:.3f}'.format(phase, flux))


In [ ]:
def phase_bin_summary(folded_points, n_bins=5):
    bins = [[] for _ in range(n_bins)]
    for phase, flux in folded_points:
        index = min(int(phase * n_bins), n_bins - 1)
        bins[index].append(flux)
    summary = []
    for index, values in enumerate(bins):
        center = (index + 0.5) / n_bins
        mean_flux = sum(values) / len(values) if values else None
        summary.append((center, None if mean_flux is None else round(mean_flux, 3), len(values)))
    return summary

print('Phase-bin summary for best period:')
for item in phase_bin_summary(folded, n_bins=6):
    print('  ', item)


In [ ]:
alternative_period = solutions[1]['period']
alternative_folded = phase_fold(rows, alternative_period)
print('Best vs. second-best candidate periods:')
print('  best period =', round(best['period'], 3), 'day')
print('  second-best period =', round(alternative_period, 3), 'day')
print('  second-best first five folded points:')
for phase, flux in alternative_folded[:5]:
    print('   phase={:.3f}, flux={:.3f}'.format(phase, flux))


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print('matplotlib unavailable; skipped time-series plots:', exc)
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].errorbar(times, fluxes, yerr=errors, fmt='o', color='tab:blue', capsize=2)
    axes[0].set_title('Light Curve')
    axes[0].set_xlabel('Time [day]')
    axes[0].set_ylabel('Normalized Flux')
    phases = [phase for phase, _ in folded]
    folded_fluxes = [flux for _, flux in folded]
    axes[1].scatter(phases, folded_fluxes, color='tab:orange')
    axes[1].set_title('Phase Folded Curve')
    axes[1].set_xlabel('Phase')
    axes[1].set_ylabel('Normalized Flux')
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
    print('Displayed raw and phase-folded views.')


## 解释

这个最小示例展示了周期恢复的核心骨架：对每个候选周期做一个简化正弦拟合，用加权残差比较好坏，再把最佳候选拿去相位折叠。真正重要的不是把一个数字报出来，而是理解为什么某些候选周期能把观测点对齐，而另一些只是在离散采样下制造了混叠幻觉。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'time_span_day': (min(times), max(times)),
    'best_period_day': round(best['period'], 3),
    'best_chi2': round(best['chi2'], 3),
    'best_amplitude': round(best['amplitude'], 4),
    'top_periods': [round(candidate['period'], 3) for candidate in solutions[:3]],
    'second_best_period_day': round(solutions[1]['period'], 3),
    'python_version': platform.python_version(),
}

print('Time-series snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')


## V2 Evidence Record artifact

本章的记录重点是采样、周期搜索范围、最佳与次优周期，以及 alias 风险。


In [ ]:
artifact_dir = Path('results')
artifact_dir.mkdir(parents=True, exist_ok=True)
evidence_path = artifact_dir / 'ch14_evidence_record.md'

cadences = [round(times[index + 1] - times[index], 3) for index in range(len(times) - 1)]
missing_pattern = 'irregular cadence' if len(set(cadences)) > 1 else 'regular cadence'
period_search_range = (min(candidate_periods), max(candidate_periods))
second_best = solutions[1]
alias_risk = abs(second_best['period'] - best['period']) <= 0.1
phase_summary = phase_bin_summary(folded, n_bins=6)

record_text = f"""# Evidence Record - Ch14 Time Series and Periods

## Record Type

- Record type: measurement / model fit

## 1. Input

- Data / file path, preferably relative to project root: `{DATA_PATH}`
- Data source or generation method: AIforAstronomers teaching light curve
- Script / notebook: `ch14_time_series_and_periods.ipynb`
- Code version / tag, if relevant: fill in the current Git commit or release tag when submitting

## 2. Operation

- Command or entry point: run notebook cells in order
- Key parameters: period grid `{period_search_range[0]:.2f}` to `{period_search_range[1]:.2f}` day in `{candidate_periods[1] - candidate_periods[0]:.2f}` day steps; weighted sine fit
- Random seed, if relevant: not applicable
- Output directory: `results/`

## 3. Output

- Output file(s): `{evidence_path}`; optional display plot if matplotlib is available
- Short result summary: best period `{best['period']:.3f}` day; second-best `{second_best['period']:.3f}` day; best chi-square `{best['chi2']:.3f}`
- Generated by which script/notebook: `ch14_time_series_and_periods.ipynb`

## 4. Check

- Check performed: compared top period candidates, phase-folded best and second-best periods, summarized phase bins
- Evidence from the check: top periods `{[round(candidate['period'], 3) for candidate in solutions[:5]]}`; phase-bin summary `{phase_summary}`; alias risk close to best = `{alias_risk}`

## 5. Limit

- Known limitation: period grid is coarse and the model is a single sinusoid; no Lomb-Scargle implementation or window-function analysis is included
- Selection / quality / uncertainty issue, if relevant: small teaching light curve with irregular cadence and no explicit observing-window metadata

## 6. Reuse in ML

- Sample / feature / label: a folded light curve or period/amplitude summary could become features; object class could become a label
- Uncertainty / quality flag: flux errors are used as weights but not propagated into a period posterior
- Preprocessing record: records period grid, phase folding, and candidate comparison
- Reproducibility evidence: records source table, period search settings, best/second-best candidates, and alias warning

## Ch14 Fields

- time series file: `{DATA_PATH}`
- time unit/system: day; no absolute time system metadata in teaching table
- cadence: irregular; cadence range `{(min(cadences), max(cadences))}` day
- duration: `{max(times) - min(times):.3f}` day
- missing pattern: {missing_pattern}
- period search range: `{period_search_range}` day
- best period: `{best['period']:.3f}` day
- second-best period: `{second_best['period']:.3f}` day
- phase-folded plot: displayed if matplotlib is available; phase-bin summary recorded here
- alias risk: `{alias_risk}` because second-best period is close to the best candidate
"""

evidence_path.write_text(record_text, encoding='utf-8')
print('wrote Evidence Record:', evidence_path)
